### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [ ]:
%pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [5]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [6]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [7]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [8]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [9]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [10]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [11]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [14]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [16]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [17]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [18]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [19]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [20]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [21]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [22]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [23]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750])

Obtenemos los 5 documentos más similares:

In [24]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [25]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [26]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [27]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [28]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [29]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## 1. Respuesta - Vectorizar documentos

In [30]:
np.random.seed(42)
indices = np.random.choice(len(newsgroups_train.data), size=5, replace=False)
print(indices)

[7492 3546 5582 4793 3813]


In [31]:
for idx in indices:
    cossim = cosine_similarity(X_train[idx], X_train)[0]
    most_similar = np.argsort(cossim)[::-1][1:6]

    print("=" * 80)
    print(f"DOCUMENTO {idx}")
    print(f"Categoría: {newsgroups_train.target_names[y_train[idx]]}")
    print(f"Texto (primeros 200 chars): {newsgroups_train.data[idx][:200]}")
    print(f"\n  5 más similares:")

    for rank, i in enumerate(most_similar, 1):
        print(f"  {rank}. Doc {i} | sim={cossim[i]:.3f} | cat: {newsgroups_train.target_names[y_train[i]]}")
    print()

DOCUMENTO 7492
Categoría: comp.sys.mac.hardware
Texto (primeros 200 chars): Could someone please post any info on these systems.

Thanks.
BoB
-- 
---------------------------------------------------------------------- 
Robert Novitskey | "Pursuing women is similar to banging o

  5 más similares:
  1. Doc 10935 | sim=0.667 | cat: comp.sys.mac.hardware
  2. Doc 7258 | sim=0.348 | cat: comp.sys.ibm.pc.hardware
  3. Doc 4971 | sim=0.180 | cat: comp.sys.mac.hardware
  4. Doc 4303 | sim=0.155 | cat: misc.forsale
  5. Doc 645 | sim=0.141 | cat: comp.sys.mac.hardware

DOCUMENTO 3546
Categoría: comp.os.ms-windows.misc
Texto (primeros 200 chars): 

     Don't bother if you have CPBackup or Fastback.  They all offer options 
not available in the stripped-down MS version (FROM CPS!).  Examples - no 
proprietary format (to save space), probably n

  5 más similares:
  1. Doc 5665 | sim=0.204 | cat: comp.sys.ibm.pc.hardware
  2. Doc 2011 | sim=0.192 | cat: comp.sys.ibm.pc.hardware
  3. Doc 8643 | si

### Documento 7492 - comp.sys.mac.hardware
El más similar (sim=0.667) es de la misma categoría, y el segundo es comp.sys.ibm.pc.hardware. Tiene sentido: ambas categorías hablan de hardware de computadoras, comparten vocabulario (systems, info, etc.). El de misc.forsale probablemente sea alguien vendiendo hardware de Mac.

### Documento 3546 - comp.os.ms-windows.misc
Los 5 más similares son todos comp.sys.ibm.pc.hardware. Ninguno es de su misma categoría, pero tiene mucho sentido: el texto habla de backups, CPBackup, software de Windows que corre en PCs IBM. Comparten vocabulario técnico de PC. Las similaridades son bajas (0.16-0.20), lo que indica que no hay documentos muy parecidos.

### Documento 5582 - misc.forsale
3 de los 5 más similares son misc.forsale y los otros 2 son comp.graphics. Tiene sentido: el texto vende hardware (disk drive, motherboard, monitor), vocabulario que se cruza con computación gráfica. Buena similaridad.

### Documento 4793 - talk.politics.guns
Resultado más mezclado: guns, crypt, politics.misc, atheism. Tiene sentido parcial: habla de armas en parques nacionales de Canadá, lo que toca temas de política y leyes. sci.crypt podría compartir vocabulario sobre regulación/gobierno. alt.atheism es más raro pero las similaridades son todas bajas (~0.23), lo que indica que no hay documentos muy parecidos.

### Documento 3813 - rec.sport.hockey

Habla de una máscara de hockey (Richter's mask, the Beezer). Pero los 5 más similares son alt.atheism, soc.religion.christian y sci.crypt. Nada de hockey. Las simiiliridades son muy bajas, indicando que no hay documentos parecidos y los encontrados son los menos distintos.

Observación general: cuando las similaridades son altas (>0.5), los documentos tienden a ser de la misma categoría. Cuando son bajas (<0.3), aparece más mezcla entre categorías, lo cual es esperable.

## 2. Respuesta - Construir un modelo de clasificación por prototipos (tipo zero-shot).

In [32]:
sim_matrix = cosine_similarity(X_test, X_train)
most_similar_train = np.argmax(sim_matrix, axis=1)

# La predicción es la etiqueta del doc de train más similar
y_pred_proto = y_train[most_similar_train]

# Evaluamos con F1 macro
f1 = f1_score(y_test, y_pred_proto, average='macro')
print(f"F1-Score Macro (clasificación por prototipos): {f1:.4f}")

F1-Score Macro (clasificación por prototipos): 0.5050


El clasificador por prototipos obtuvo un F1-Score Macro de 0.5050. Considerando que es un método que no entrena ningún modelo y que clasifica entre 20 categorías (donde el azar daría ~0.05), el resultado es razonable. Sin embargo, tiene limitaciones: es costoso computacionalmente (compara cada documento de test contra todos los de train) y su desempeño depende de que exista un documento de train lo suficientemente similar. Como vimos en el punto 1, documentos con vocabulario poco específico generan similitudes bajas, lo que lleva a clasificaciones incorrectas.



## 3. Respuesta - Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación

In [38]:
# === BASELINE ===
vectorizador_baseline = TfidfVectorizer()
X_train_baseline = vectorizador_baseline.fit_transform(newsgroups_train.data)
X_test_baseline = vectorizador_baseline.transform(newsgroups_test.data)

modelo_baseline = MultinomialNB()
modelo_baseline.fit(X_train_baseline, y_train)
predicciones_baseline = modelo_baseline.predict(X_test_baseline)
f1_baseline = f1_score(y_test, predicciones_baseline, average='macro')
print(f"Baseline - MultinomialNB + TfidfVectorizer: {f1_baseline:.4f}")

# === EXPERIMENTO 1: sublinear_tf suaviza palabras muy repetidas ===
vectorizador_sublinear = TfidfVectorizer(sublinear_tf=True)
X_train_sublinear = vectorizador_sublinear.fit_transform(newsgroups_train.data)
X_test_sublinear = vectorizador_sublinear.transform(newsgroups_test.data)

modelo_multinomial_sublinear = MultinomialNB()
modelo_multinomial_sublinear.fit(X_train_sublinear, y_train)
predicciones_multinomial_sublinear = modelo_multinomial_sublinear.predict(X_test_sublinear)
print(f"MultinomialNB + sublinear_tf: {f1_score(y_test, predicciones_multinomial_sublinear, average='macro'):.4f}")

# === EXPERIMENTO 2: ComplementNB ===
modelo_complement = ComplementNB()
modelo_complement.fit(X_train_baseline, y_train)
predicciones_complement = modelo_complement.predict(X_test_baseline)
print(f"ComplementNB: {f1_score(y_test, predicciones_complement, average='macro'):.4f}")

# === EXPERIMENTO 3: ComplementNB + sublinear_tf ===
modelo_complement_sublinear = ComplementNB()
modelo_complement_sublinear.fit(X_train_sublinear, y_train)
predicciones_complement_sublinear = modelo_complement_sublinear.predict(X_test_sublinear)
print(f"ComplementNB + sublinear_tf: {f1_score(y_test, predicciones_complement_sublinear, average='macro'):.4f}")

# === EXPERIMENTO 4: ComplementNB variando alpha ===
print("\n--- Variando alpha ---")
for alpha in [1.0, 0.5, 0.1, 0.01]:
    modelo_alpha = ComplementNB(alpha=alpha)
    modelo_alpha.fit(X_train_baseline, y_train)
    predicciones_alpha = modelo_alpha.predict(X_test_baseline)
    print(f"ComplementNB alpha={alpha}: {f1_score(y_test, predicciones_alpha, average='macro'):.4f}")

# === EXPERIMENTO 5: ComplementNB filtrando por frecuencia ===
print("\n--- Filtrando por frecuencia ---")
for max_df, min_df in [(0.9, 2), (0.8, 3), (0.7, 5), (0.5, 5)]:
    vectorizador_filtrado = TfidfVectorizer(max_df=max_df, min_df=min_df)
    X_train_filtrado = vectorizador_filtrado.fit_transform(newsgroups_train.data)
    X_test_filtrado = vectorizador_filtrado.transform(newsgroups_test.data)
    modelo_filtrado = ComplementNB(alpha=0.1)
    modelo_filtrado.fit(X_train_filtrado, y_train)
    predicciones_filtrado = modelo_filtrado.predict(X_test_filtrado)
    print(f"ComplementNB max_df={max_df} min_df={min_df}: {f1_score(y_test, predicciones_filtrado, average='macro'):.4f}")

# === EXPERIMENTO 6: MEJOR COMBINACIÓN - ComplementNB alpha=0.5 + sublinear_tf ===
vectorizador_mejor = TfidfVectorizer(sublinear_tf=True)
X_train_mejor = vectorizador_mejor.fit_transform(newsgroups_train.data)
X_test_mejor = vectorizador_mejor.transform(newsgroups_test.data)

modelo_mejor = ComplementNB(alpha=0.5)
modelo_mejor.fit(X_train_mejor, y_train)
predicciones_mejor = modelo_mejor.predict(X_test_mejor)
print(f"\nMejor modelo - ComplementNB alpha=0.5 + sublinear_tf: {f1_score(y_test, predicciones_mejor, average='macro'):.4f}")

# === EXPERIMENTO 7: CountVectorizer (sin IDF) ===
print("\n--- CountVectorizer (sin IDF) ---")
vectorizador_conteo = CountVectorizer()
X_train_conteo = vectorizador_conteo.fit_transform(newsgroups_train.data)
X_test_conteo = vectorizador_conteo.transform(newsgroups_test.data)

for alpha in [1.0, 0.5, 0.1]:
    modelo_conteo = ComplementNB(alpha=alpha)
    modelo_conteo.fit(X_train_conteo, y_train)
    predicciones_conteo = modelo_conteo.predict(X_test_conteo)
    print(f"CountVectorizer + ComplementNB alpha={alpha}: {f1_score(y_test, predicciones_conteo, average='macro'):.4f}")

Baseline - MultinomialNB + TfidfVectorizer: 0.5854
MultinomialNB + sublinear_tf: 0.5860
ComplementNB: 0.6930
ComplementNB + sublinear_tf: 0.6920

--- Variando alpha ---
ComplementNB alpha=1.0: 0.6930
ComplementNB alpha=0.5: 0.6961
ComplementNB alpha=0.1: 0.6954
ComplementNB alpha=0.01: 0.6689

--- Filtrando por frecuencia ---
ComplementNB max_df=0.9 min_df=2: 0.6906
ComplementNB max_df=0.8 min_df=3: 0.6841
ComplementNB max_df=0.7 min_df=5: 0.6763
ComplementNB max_df=0.5 min_df=5: 0.6764

Mejor modelo - ComplementNB alpha=0.5 + sublinear_tf: 0.6968

--- CountVectorizer (sin IDF) ---
CountVectorizer + ComplementNB alpha=1.0: 0.6374
CountVectorizer + ComplementNB alpha=0.5: 0.6404
CountVectorizer + ComplementNB alpha=0.1: 0.6386


El mejor modelo fue ComplementNB con alpha=0.5 y TfidfVectorizer con sublinear_tf=True, obteniendo un F1-Score Macro de 0.6968. Esto representa una mejora de +11 puntos respecto al baseline (0.5854) y +19 puntos respecto al clasificador por prototipos (0.5050). Los factores que más impactaron fueron: (1) usar ComplementNB en vez de MultinomialNB (+10 puntos), lo cual es esperado en datasets con clases de distintos tamaños; (2) ajustar alpha a 0.5 para un suavizado intermedio; (3) sublinear_tf aportó una mejora marginal. CountVectorizer (sin IDF) empeoró los resultados, confirmando que la ponderación IDF es importante para esta tarea. Los filtros max_df/min_df no mejoraron el desempeño.

## Respuesta 4 - Transponer la matriz documento-término.

In [39]:
matriz_termino_documento = X_train.T

print(f"Matriz documento-término: {X_train.shape}")
print(f"Matriz término-documento: {matriz_termino_documento.shape}")

palabras_elegidas = ['car', 'gun', 'god', 'space', 'hockey']
indices_palabras = [tfidfvect.vocabulary_[p] for p in palabras_elegidas]

for palabra, idx_palabra in zip(palabras_elegidas, indices_palabras):
    similaridad_palabra = cosine_similarity(matriz_termino_documento[idx_palabra], matriz_termino_documento)[0]

    palabras_mas_similares = np.argsort(similaridad_palabra)[::-1][1:6]

    print(f"\n'{palabra}' - 5 palabras más similares:")
    for i in palabras_mas_similares:
        print(f"  {idx2word[i]:15s} | sim={similaridad_palabra[i]:.3f}")

Matriz documento-término: (11314, 101631)
Matriz término-documento: (101631, 11314)

'car' - 5 palabras más similares:
  cars            | sim=0.180
  criterium       | sim=0.177
  civic           | sim=0.175
  owner           | sim=0.169
  dealer          | sim=0.168

'gun' - 5 palabras más similares:
  guns            | sim=0.358
  crime           | sim=0.244
  handgun         | sim=0.239
  homicides       | sim=0.233
  firearms        | sim=0.233

'god' - 5 palabras más similares:
  jesus           | sim=0.269
  bible           | sim=0.262
  that            | sim=0.256
  existence       | sim=0.255
  christ          | sim=0.251

'space' - 5 palabras más similares:
  nasa            | sim=0.330
  seds            | sim=0.297
  shuttle         | sim=0.293
  enfant          | sim=0.280
  seti            | sim=0.246

'hockey' - 5 palabras más similares:
  ncaa            | sim=0.274
  nhl             | sim=0.265
  affiliates      | sim=0.248
  xenophobes      | sim=0.243
  sportschannel 

La transposición de la matriz documento-término permite obtener representaciones vectoriales de palabras basadas en co-ocurrencia en documentos. Los resultados muestran que las palabras más similares son en general semánticamente coherentes: "car" se asocia con "cars", "civic", "dealer"; "gun" con "firearms", "crime", "handgun"; "god" con "jesus", "bible", "christ"; "space" con "nasa", "shuttle", "seti"; y "hockey" con "nhl", "ncaa". Sin embargo, aparecen algunas palabras ruidosas como "that", "criterium" y "xenophobes" que no tienen relación semántica evidente, lo cual es una limitación de este enfoque basado únicamente en co-ocurrencia documental.

